# Fix Missing Weather Data

This notebook identifies and fetches missing weather data from the Open-Meteo API.

**Objective**: 
- Analyze existing weather_data.csv to identify missing entries
- Fetch missing weather data one by one to handle timeouts
- Update the weather dataset with complete coverage

**Approach**:
1. Load existing weather data and flight data
2. Identify missing weather records 
3. Fetch missing data with retry logic and rate limiting
4. Merge and save updated dataset

In [21]:
# Import required libraries
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta
from pathlib import Path
import json
from typing import Dict, List, Optional, Tuple
import warnings

warnings.filterwarnings('ignore')

print("📚 Libraries imported successfully")
print(f"🕐 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Data paths
data_path = Path('../data')
print(f"📁 Data directory: {data_path}")

# API settings
OPENMETEO_API_BASE = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_DELAY = 1.1  # Seconds between requests (to stay under rate limit)
MAX_RETRIES = 3
TIMEOUT = 30

print(f"🌤️ Open-Meteo API settings:")
print(f"   • Base URL: {OPENMETEO_API_BASE}")
print(f"   • Request delay: {REQUEST_DELAY}s")
print(f"   • Max retries: {MAX_RETRIES}")
print(f"   • Timeout: {TIMEOUT}s")

📚 Libraries imported successfully
🕐 Current time: 2025-09-21 22:50:11
📁 Data directory: ..\data
🌤️ Open-Meteo API settings:
   • Base URL: https://archive-api.open-meteo.com/v1/archive
   • Request delay: 1.1s
   • Max retries: 3
   • Timeout: 30s


In [22]:
# Load existing data
print("📊 LOADING EXISTING DATA")
print("=" * 40)

# Load weather data
weather_file = data_path / 'weather_data.csv'
if weather_file.exists():
    df_weather = pd.read_csv(weather_file)
    print(f"✅ Loaded existing weather data: {len(df_weather):,} records")
    
    # Convert datetime columns
    df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])
    
    print(f"📅 Weather data date range: {df_weather['datetime'].min()} to {df_weather['datetime'].max()}")
    print(f"🌍 Airports with weather data: {df_weather['airport'].nunique()}")
    print(f"📊 Weather data overview:")
    print(df_weather.info())
else:
    print("❌ No existing weather data found!")
    df_weather = pd.DataFrame()

# Load historical flight data to understand what we need
flights_file = data_path / 'historical_flights.csv'
if flights_file.exists():
    print(f"\n📈 Loading flight data to identify required weather records...")
    df_flights = pd.read_csv(flights_file)
    df_flights['std'] = pd.to_datetime(df_flights['std'])
    
    print(f"✅ Loaded flight data: {len(df_flights):,} records")
    print(f"📅 Flight date range: {df_flights['std'].min()} to {df_flights['std'].max()}")
    print(f"✈️ Departure airports: {df_flights['dep_airport'].nunique()}")
else:
    print("❌ No flight data found!")
    df_flights = pd.DataFrame()

# Load airport coordinates
airport_coords_file = data_path / 'airport_coordinates.csv'
if airport_coords_file.exists():
    df_airports = pd.read_csv(airport_coords_file)
    print(f"\n🗺️ Loaded airport coordinates: {len(df_airports)} airports")
    print(f"   • Airports with coordinates: {df_airports.dropna(subset=['latitude', 'longitude']).shape[0]}")
else:
    print("❌ No airport coordinates found!")
    df_airports = pd.DataFrame()

📊 LOADING EXISTING DATA
✅ Loaded existing weather data: 335,839 records
📅 Weather data date range: 2018-01-01 08:00:00 to 2025-07-31 22:00:00
🌍 Airports with weather data: 121
📊 Weather data overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335839 entries, 0 to 335838
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   datetime              335839 non-null  datetime64[ns]
 1   temperature_c         186849 non-null  float64       
 2   humidity_percent      186849 non-null  float64       
 3   precipitation_mm      186849 non-null  float64       
 4   pressure_hpa          186849 non-null  float64       
 5   cloud_cover_percent   186849 non-null  float64       
 6   wind_speed_kmh        186849 non-null  float64       
 7   wind_direction_deg    186849 non-null  float64       
 8   airport               335839 non-null  object        
 9   latitude              335839 non-n

In [23]:
# Identify missing weather data
print("🔍 ANALYZING MISSING WEATHER DATA")
print("=" * 45)

if not df_flights.empty and not df_airports.empty:
    # Create required weather records based on flight data
    print("📋 Creating list of required weather records...")
    
    # Get unique combinations of airport and date (rounded to hour) from flights
    df_flights['date_hour'] = df_flights['std'].dt.floor('H')
    
    required_records = df_flights[['dep_airport', 'date_hour']].drop_duplicates()
    required_records = required_records.rename(columns={'dep_airport': 'airport', 'date_hour': 'datetime'})
    
    print(f"📊 Required weather records: {len(required_records):,}")
    print(f"📅 Date range needed: {required_records['datetime'].min()} to {required_records['datetime'].max()}")
    print(f"✈️ Airports needing weather: {required_records['airport'].nunique()}")
    
    if not df_weather.empty:
        # Find missing records
        print(f"\n🔎 Comparing with existing weather data...")
        
        # Create a key for comparison
        required_records['key'] = required_records['airport'] + '_' + required_records['datetime'].dt.strftime('%Y-%m-%d_%H')
        
        # Standardize column names - rename airport_code to airport for consistency
        if 'airport_code' in df_weather.columns:
            df_weather = df_weather.rename(columns={'airport_code': 'airport'})
            print(f"🔧 Renamed 'airport_code' to 'airport' for consistency")
        
        df_weather['key'] = df_weather['airport'] + '_' + pd.to_datetime(df_weather['datetime']).dt.strftime('%Y-%m-%d_%H')
        
        existing_keys = set(df_weather['key'].unique())
        required_keys = set(required_records['key'].unique())
        
        missing_keys = required_keys - existing_keys
        missing_records = required_records[required_records['key'].isin(missing_keys)].copy()
        
        print(f"✅ Existing weather records: {len(existing_keys):,}")
        print(f"❌ Missing weather records: {len(missing_records):,}")
        print(f"📈 Coverage: {len(existing_keys)/(len(existing_keys)+len(missing_records))*100:.1f}%")
        
        # Add coordinates to missing records
        missing_records = missing_records.merge(
            df_airports[['iata_code', 'latitude', 'longitude']], 
            left_on='airport', 
            right_on='iata_code', 
            how='left'
        )
        
        # Remove records without coordinates
        missing_with_coords = missing_records.dropna(subset=['latitude', 'longitude'])
        missing_no_coords = missing_records[missing_records[['latitude', 'longitude']].isnull().any(axis=1)]
        
        print(f"\n🗺️ Missing records with coordinates: {len(missing_with_coords):,}")
        print(f"❓ Missing records without coordinates: {len(missing_no_coords):,}")
        
        if len(missing_no_coords) > 0:
            print(f"⚠️ Airports without coordinates: {missing_no_coords['airport'].unique()}")
        
        # Show sample of missing records
        print(f"\n📋 Sample missing records:")
        if len(missing_with_coords) > 0:
            sample_missing = missing_with_coords.head(10)[['airport', 'datetime', 'latitude', 'longitude']]
            print(sample_missing.to_string(index=False))
        
        missing_to_fetch = missing_with_coords
        
    else:
        print(f"ℹ️ No existing weather data - need to fetch all records")
        missing_to_fetch = required_records.merge(
            df_airports[['iata_code', 'latitude', 'longitude']], 
            left_on='airport', 
            right_on='iata_code', 
            how='left'
        ).dropna(subset=['latitude', 'longitude'])
    
    print(f"\n🎯 Total records to fetch: {len(missing_to_fetch):,}")
    
else:
    print("❌ Cannot analyze missing data - flight data or airport coordinates missing")
    missing_to_fetch = pd.DataFrame()

🔍 ANALYZING MISSING WEATHER DATA
📋 Creating list of required weather records...
📊 Required weather records: 333,730
📅 Date range needed: 2018-01-01 07:00:00 to 2025-07-31 21:00:00
✈️ Airports needing weather: 118

🔎 Comparing with existing weather data...
✅ Existing weather records: 335,839
❌ Missing weather records: 58,824
📈 Coverage: 85.1%

🗺️ Missing records with coordinates: 58,824
❓ Missing records without coordinates: 0

📋 Sample missing records:
airport            datetime  latitude  longitude
    SSJ 2019-06-12 16:00:00 65.958065  12.471378
    SVJ 2020-03-18 12:00:00 68.241702  14.667082
    SVJ 2020-03-19 12:00:00 68.241702  14.667082
    SVJ 2020-03-20 12:00:00 68.241702  14.667082
    SVJ 2020-03-22 12:00:00 68.241702  14.667082
    SVJ 2020-03-23 12:00:00 68.241702  14.667082
    SSJ 2018-01-01 12:00:00 65.958065  12.471378
    SSJ 2018-01-01 17:00:00 65.958065  12.471378
    TRD 2018-01-01 18:00:00 63.454766  10.925133
    SSJ 2018-01-01 19:00:00 65.958065  12.471378

🎯 T

In [24]:
# Weather fetching functions
print("🌤️ SETTING UP WEATHER FETCHING FUNCTIONS")
print("=" * 50)

def fetch_single_weather_record(airport: str, latitude: float, longitude: float, datetime_str: str, 
                               retry_count: int = 0) -> Optional[Dict]:
    """
    Fetch weather data for a single airport and datetime with retry logic
    """
    try:
        # Parse datetime
        dt = pd.to_datetime(datetime_str)
        date_str = dt.strftime('%Y-%m-%d')
        
        # Prepare API request
        params = {
            'latitude': latitude,
            'longitude': longitude,
            'start_date': date_str,
            'end_date': date_str,
            'hourly': [
                'temperature_2m',
                'relative_humidity_2m', 
                'precipitation',
                'surface_pressure',
                'cloud_cover',
                'wind_speed_10m',
                'wind_direction_10m',
                'visibility'
            ],
            'timezone': 'UTC'
        }
        
        # Make API request with timeout
        response = requests.get(OPENMETEO_API_BASE, params=params, timeout=TIMEOUT)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'hourly' in data and data['hourly']:
                # Find the closest hour
                hourly_times = pd.to_datetime(data['hourly']['time'])
                target_hour = dt.floor('H')
                
                # Find exact match or closest
                time_diffs = abs((hourly_times - target_hour).total_seconds())
                closest_idx = time_diffs.argmin()
                
                # Extract weather data for the closest hour
                weather_record = {
                    'airport': airport,
                    'datetime': target_hour,
                    'latitude': latitude,
                    'longitude': longitude,
                    'temperature_2m': data['hourly']['temperature_2m'][closest_idx],
                    'relative_humidity_2m': data['hourly']['relative_humidity_2m'][closest_idx],
                    'precipitation': data['hourly']['precipitation'][closest_idx],
                    'surface_pressure': data['hourly']['surface_pressure'][closest_idx],
                    'cloud_cover': data['hourly']['cloud_cover'][closest_idx],
                    'wind_speed_10m': data['hourly']['wind_speed_10m'][closest_idx],
                    'wind_direction_10m': data['hourly']['wind_direction_10m'][closest_idx],
                    'visibility': data['hourly']['visibility'][closest_idx]
                }
                
                return weather_record
            else:
                print(f"⚠️ No hourly data for {airport} at {datetime_str}")
                return None
                
        elif response.status_code == 429:  # Rate limit
            if retry_count < MAX_RETRIES:
                wait_time = 2 ** retry_count  # Exponential backoff
                print(f"⏳ Rate limited for {airport} - waiting {wait_time}s before retry {retry_count + 1}")
                time.sleep(wait_time)
                return fetch_single_weather_record(airport, latitude, longitude, datetime_str, retry_count + 1)
            else:
                print(f"❌ Rate limit exceeded for {airport} at {datetime_str}")
                return None
        else:
            print(f"❌ API error {response.status_code} for {airport} at {datetime_str}")
            return None
            
    except requests.exceptions.Timeout:
        if retry_count < MAX_RETRIES:
            print(f"⏳ Timeout for {airport} - retry {retry_count + 1}")
            time.sleep(2)
            return fetch_single_weather_record(airport, latitude, longitude, datetime_str, retry_count + 1)
        else:
            print(f"❌ Timeout exceeded for {airport} at {datetime_str}")
            return None
    except Exception as e:
        print(f"❌ Error fetching weather for {airport} at {datetime_str}: {str(e)}")
        return None

def save_progress(weather_records: List[Dict], filename: str = 'weather_data_progress.csv'):
    """Save progress to avoid losing data"""
    if weather_records:
        progress_df = pd.DataFrame(weather_records)
        progress_file = data_path / filename
        progress_df.to_csv(progress_file, index=False)
        print(f"💾 Progress saved: {len(weather_records)} records to {filename}")

print("✅ Weather fetching functions ready")
print(f"🔧 Functions available:")
print(f"   • fetch_single_weather_record() - Fetch one record with retry logic")
print(f"   • save_progress() - Save intermediate results")

🌤️ SETTING UP WEATHER FETCHING FUNCTIONS
✅ Weather fetching functions ready
🔧 Functions available:
   • fetch_single_weather_record() - Fetch one record with retry logic
   • save_progress() - Save intermediate results


In [25]:
# Fetch missing weather data (optimized using batch per-airport approach)
print("🚀 FETCHING MISSING WEATHER DATA (OPTIMIZED)")
print("=" * 40)

if not missing_to_fetch.empty:
    # Prepare keys and grouping
    missing_to_fetch['datetime'] = pd.to_datetime(missing_to_fetch['datetime'])
    missing_to_fetch['datetime_hour'] = missing_to_fetch['datetime'].dt.floor('H')
    missing_to_fetch['date'] = missing_to_fetch['datetime_hour'].dt.date

    # Group by airport -> list of required hours
    airport_groups = missing_to_fetch.groupby('airport')

    print(f"📊 Airports with missing records: {len(airport_groups)}")

    fetched_records = []
    failed_airports = []
    progress_file = data_path / 'weather_data_progress.csv'
    start_time = datetime.now()

    # Load existing progress (if any) to avoid refetching
    processed_keys = set()
    if progress_file.exists():
        prev = pd.read_csv(progress_file)
        prev['datetime'] = pd.to_datetime(prev['datetime'])
        prev_keys = prev['airport'] + '_' + prev['datetime'].dt.strftime('%Y-%m-%d_%H')
        processed_keys = set(prev_keys.unique())
        fetched_records = prev.to_dict('records')
        print(f"📋 Loaded existing progress: {len(prev):,} records")

    # Helper: fetch date-range weather for an airport (returns DataFrame)
    def fetch_weather_batch(airport_code, lat, lon, start_date, end_date):
        try:
            params = {
                'latitude': lat,
                'longitude': lon,
                'start_date': start_date.strftime('%Y-%m-%d'),
                'end_date': end_date.strftime('%Y-%m-%d'),
                'hourly': [
                    'temperature_2m', 'relative_humidity_2m', 'precipitation',
                    'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'visibility'
                ],
                'timezone': 'UTC'
            }
            resp = requests.get(OPENMETEO_API_BASE, params=params, timeout=TIMEOUT)
            if resp.status_code != 200:
                return pd.DataFrame()
            data = resp.json()
            hourly = data.get('hourly', {})
            times = hourly.get('time', [])
            if not times:
                return pd.DataFrame()

            df = pd.DataFrame({
                'datetime': pd.to_datetime(times),
                'temperature_2m': hourly.get('temperature_2m', [None] * len(times)),
                'relative_humidity_2m': hourly.get('relative_humidity_2m', [None] * len(times)),
                'precipitation': hourly.get('precipitation', [None] * len(times)),
                'surface_pressure': hourly.get('surface_pressure', [None] * len(times)),
                'cloud_cover': hourly.get('cloud_cover', [None] * len(times)),
                'wind_speed_10m': hourly.get('wind_speed_10m', [None] * len(times)),
                'wind_direction_10m': hourly.get('wind_direction_10m', [None] * len(times)),
                'visibility': hourly.get('visibility', [None] * len(times))
            })
            df['airport'] = airport_code
            df['latitude'] = lat
            df['longitude'] = lon
            return df
        except Exception:
            return pd.DataFrame()

    # Iterate airports and fetch ranges in batch
    total_airports = len(airport_groups)
    processed = 0
    save_every = 50  # save progress every N airports

    # Build coords lookup
    coords_lookup = df_airports.set_index('iata_code')[['latitude', 'longitude']].to_dict('index')

    for airport_code, group in airport_groups:
        processed += 1
        try:
            if airport_code not in coords_lookup:
                failed_airports.append(airport_code)
                continue

            lat = coords_lookup[airport_code]['latitude']
            lon = coords_lookup[airport_code]['longitude']

            # Determine date range to fetch (min to max date for required hours)
            dates = sorted(set(group['date']))
            start_date = min(dates)
            end_date = max(dates)

            # Fetch full date range in single request
            weather_df = fetch_weather_batch(airport_code, lat, lon, pd.to_datetime(start_date), pd.to_datetime(end_date))

            if weather_df.empty:
                failed_airports.append(airport_code)
            else:
                # Keep only hours we need
                needed_hours = set(group['datetime_hour'].dt.strftime('%Y-%m-%d %H:00:00'))
                weather_df['datetime_hour'] = weather_df['datetime'].dt.floor('H')
                weather_df_filtered = weather_df[weather_df['datetime_hour'].dt.strftime('%Y-%m-%d %H:00:00').isin(needed_hours)].copy()

                if not weather_df_filtered.empty:
                    # Convert rows to records and append only keys not already processed
                    for _, row in weather_df_filtered.iterrows():
                        key = f"{row['airport']}_{row['datetime_hour'].strftime('%Y-%m-%d_%H')}"
                        if key in processed_keys:
                            continue
                        rec = {
                            'airport': row['airport'],
                            'datetime': row['datetime_hour'],
                            'latitude': row['latitude'],
                            'longitude': row['longitude'],
                            'temperature_2m': row.get('temperature_2m'),
                            'relative_humidity_2m': row.get('relative_humidity_2m'),
                            'precipitation': row.get('precipitation'),
                            'surface_pressure': row.get('surface_pressure'),
                            'cloud_cover': row.get('cloud_cover'),
                            'wind_speed_10m': row.get('wind_speed_10m'),
                            'wind_direction_10m': row.get('wind_direction_10m'),
                            'visibility': row.get('visibility')
                        }
                        fetched_records.append(rec)
                        processed_keys.add(key)
                else:
                    failed_airports.append(airport_code)

            # Periodic save
            if processed % save_every == 0:
                if fetched_records:
                    save_progress(fetched_records, 'weather_data_progress.csv')
                elapsed = datetime.now() - start_time
                print(f"📈 Processed {processed}/{total_airports} airports, elapsed: {elapsed}, fetched: {len(fetched_records):,}")

            # Respect rate limiting
            time.sleep(REQUEST_DELAY)

        except Exception as e:
            failed_airports.append(airport_code)

    # Final save
    if fetched_records:
        save_progress(fetched_records, 'weather_data_progress.csv')

    total_time = datetime.now() - start_time
    print(f"\n🎉 FETCHING COMPLETED!")
    print(f"⏱️ Total time: {total_time}")
    print(f"✅ Records fetched: {len(fetched_records):,}")
    print(f"❌ Failed airports: {len(failed_airports)}")

else:
    print("ℹ️ No missing weather data to fetch")
    fetched_records = []

🚀 FETCHING MISSING WEATHER DATA (OPTIMIZED)
📊 Airports with missing records: 14
💾 Progress saved: 31891 records to weather_data_progress.csv

🎉 FETCHING COMPLETED!
⏱️ Total time: 0:00:38.671220
✅ Records fetched: 31,891
❌ Failed airports: 7
💾 Progress saved: 31891 records to weather_data_progress.csv

🎉 FETCHING COMPLETED!
⏱️ Total time: 0:00:38.671220
✅ Records fetched: 31,891
❌ Failed airports: 7


In [26]:
# Merge and save complete weather dataset
print("🔄 MERGING AND SAVING COMPLETE DATASET")
print("=" * 45)

if fetched_records:
    # Convert fetched records to DataFrame
    new_weather = pd.DataFrame(fetched_records)
    print(f"📊 New weather records: {len(new_weather):,}")
    
    # Standardize column names - existing data uses 'airport_code', new data uses 'airport'
    if not df_weather.empty and 'airport_code' in df_weather.columns:
        df_weather = df_weather.rename(columns={'airport_code': 'airport'})
        print(f"🔧 Renamed 'airport_code' to 'airport' in existing data")
    
    # Combine with existing weather data
    if not df_weather.empty:
        # Remove duplicates by creating a combined key
        existing_keys = set(df_weather['airport'] + '_' + pd.to_datetime(df_weather['datetime']).dt.strftime('%Y-%m-%d_%H'))
        new_weather['temp_key'] = new_weather['airport'] + '_' + pd.to_datetime(new_weather['datetime']).dt.strftime('%Y-%m-%d_%H')
        
        # Filter out any duplicates (shouldn't happen, but safety check)
        new_unique = new_weather[~new_weather['temp_key'].isin(existing_keys)].drop('temp_key', axis=1)
        
        # Combine datasets
        complete_weather = pd.concat([df_weather, new_unique], ignore_index=True)
        print(f"✅ Combined dataset: {len(complete_weather):,} total records")
        print(f"   • Existing: {len(df_weather):,}")
        print(f"   • New unique: {len(new_unique):,}")
        
    else:
        complete_weather = new_weather
        print(f"✅ Created new dataset: {len(complete_weather):,} records")
    
    # Sort by airport and datetime
    complete_weather = complete_weather.sort_values(['airport', 'datetime'])
    
    # Save complete dataset
    complete_file = data_path / 'weather_data_complete.csv'
    complete_weather.to_csv(complete_file, index=False)
    print(f"💾 Complete weather dataset saved to: {complete_file}")
    
    # Update the main weather file
    main_weather_file = data_path / 'weather_data.csv'
    complete_weather.to_csv(main_weather_file, index=False)
    print(f"💾 Updated main weather file: {main_weather_file}")
    
    # Dataset statistics
    print(f"\n📊 FINAL DATASET STATISTICS:")
    print(f"   • Total records: {len(complete_weather):,}")
    print(f"   • Date range: {complete_weather['datetime'].min()} to {complete_weather['datetime'].max()}")
    print(f"   • Airports covered: {complete_weather['airport'].nunique()}")
    print(f"   • Records per airport (avg): {len(complete_weather)/complete_weather['airport'].nunique():.1f}")
    
    # Coverage analysis
    if not df_flights.empty:
        # Calculate coverage improvement
        complete_weather['key'] = complete_weather['airport'] + '_' + pd.to_datetime(complete_weather['datetime']).dt.strftime('%Y-%m-%d_%H')
        new_coverage = len(set(complete_weather['key']))
        original_coverage = len(set(df_weather['key'])) if not df_weather.empty else 0
        required_total = len(required_records) if 'required_records' in locals() else new_coverage
        
        print(f"\n📈 COVERAGE IMPROVEMENT:")
        print(f"   • Original coverage: {original_coverage:,} records")
        print(f"   • New coverage: {new_coverage:,} records")
        print(f"   • Improvement: +{new_coverage - original_coverage:,} records")
        print(f"   • Final coverage: {new_coverage/required_total*100:.1f}% of required records")
    
    # Clean up progress files
    progress_file = data_path / 'weather_data_progress.csv'
    if progress_file.exists():
        progress_file.unlink()
        print(f"🗑️ Cleaned up progress file")
        
else:
    print("ℹ️ No new weather data to merge")

print(f"\n✅ Weather data fixing completed!")
print(f"🎯 Next steps:")
print(f"   • Review any failed requests in failed_weather_requests.csv")
print(f"   • Consider running again for failed records if needed")
print(f"   • Use weather_data.csv for your analysis")

🔄 MERGING AND SAVING COMPLETE DATASET
📊 New weather records: 31,891
✅ Combined dataset: 367,730 total records
   • Existing: 335,839
   • New unique: 31,891
✅ Combined dataset: 367,730 total records
   • Existing: 335,839
   • New unique: 31,891


💾 Complete weather dataset saved to: ..\data\weather_data_complete.csv
💾 Updated main weather file: ..\data\weather_data.csv

📊 FINAL DATASET STATISTICS:
   • Total records: 367,730
   • Date range: 2018-01-01 08:00:00 to 2025-07-31 22:00:00
   • Airports covered: 128
   • Records per airport (avg): 2872.9
💾 Updated main weather file: ..\data\weather_data.csv

📊 FINAL DATASET STATISTICS:
   • Total records: 367,730
   • Date range: 2018-01-01 08:00:00 to 2025-07-31 22:00:00
   • Airports covered: 128
   • Records per airport (avg): 2872.9

📈 COVERAGE IMPROVEMENT:
   • Original coverage: 335,839 records
   • New coverage: 367,730 records
   • Improvement: +31,891 records
   • Final coverage: 110.2% of required records
🗑️ Cleaned up progress file

✅ Weather data fixing completed!
🎯 Next steps:
   • Review any failed requests in failed_weather_requests.csv
   • Consider running again for failed records if needed
   • Use weather_data.csv for your analysis

📈 COVERAGE IMPROVEMENT:
   • Ori

In [27]:
# Utility functions for retry and validation
print("🔧 UTILITY FUNCTIONS")
print("=" * 25)

def retry_failed_requests(max_retry_attempts: int = 2) -> None:
    """
    Retry failed weather requests from the failed_weather_requests.csv file
    """
    failed_file = data_path / 'failed_weather_requests.csv'
    
    if not failed_file.exists():
        print("ℹ️ No failed requests file found")
        return
    
    failed_df = pd.read_csv(failed_file)
    print(f"📋 Found {len(failed_df)} failed requests to retry")
    
    retry_records = []
    still_failed = []
    
    for idx, row in failed_df.iterrows():
        print(f"🔄 Retrying {row['airport']} at {row['datetime']} (attempt {max_retry_attempts})")
        
        weather_record = fetch_single_weather_record(
            row['airport'], row['latitude'], row['longitude'], row['datetime']
        )
        
        if weather_record:
            retry_records.append(weather_record)
            print(f"✅ Retry successful for {row['airport']}")
        else:
            still_failed.append(row.to_dict())
            print(f"❌ Retry failed for {row['airport']}")
        
        time.sleep(REQUEST_DELAY)
    
    # Save retry results
    if retry_records:
        retry_df = pd.DataFrame(retry_records)
        retry_file = data_path / 'weather_data_retry.csv'
        retry_df.to_csv(retry_file, index=False)
        print(f"💾 Saved {len(retry_records)} retry successes to {retry_file}")
    
    if still_failed:
        still_failed_df = pd.DataFrame(still_failed)
        still_failed_file = data_path / 'still_failed_weather_requests.csv'
        still_failed_df.to_csv(still_failed_file, index=False)
        print(f"💾 Saved {len(still_failed)} remaining failures to {still_failed_file}")
    
    print(f"🎯 Retry results: {len(retry_records)} successes, {len(still_failed)} still failed")

def validate_weather_data() -> None:
    """
    Validate the quality of weather data
    """
    weather_file = data_path / 'weather_data.csv'
    
    if not weather_file.exists():
        print("❌ No weather data file found")
        return
    
    df = pd.read_csv(weather_file)
    print(f"🔍 WEATHER DATA VALIDATION")
    print(f"=" * 30)
    print(f"📊 Total records: {len(df):,}")
    
    # Check for missing values
    print(f"\n🔎 Missing values:")
    missing_counts = df.isnull().sum()
    for col, count in missing_counts.items():
        if count > 0:
            print(f"   • {col}: {count:,} ({count/len(df)*100:.1f}%)")
    
    # Check data ranges
    print(f"\n📏 Data ranges:")
    numeric_cols = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 
                   'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'visibility']
    
    for col in numeric_cols:
        if col in df.columns:
            print(f"   • {col}: {df[col].min():.2f} to {df[col].max():.2f}")
    
    # Check temporal coverage
    df['datetime'] = pd.to_datetime(df['datetime'])
    print(f"\n📅 Temporal coverage:")
    print(f"   • Date range: {df['datetime'].min()} to {df['datetime'].max()}")
    print(f"   • Total hours covered: {df['datetime'].nunique():,}")
    
    # Check airport coverage
    print(f"\n✈️ Airport coverage:")
    print(f"   • Airports: {df['airport'].nunique()}")
    print(f"   • Records per airport (avg): {len(df)/df['airport'].nunique():.1f}")
    
    # Top and bottom airports by record count
    airport_counts = df['airport'].value_counts()
    print(f"   • Most records: {airport_counts.head(3).to_dict()}")
    print(f"   • Least records: {airport_counts.tail(3).to_dict()}")

print("✅ Utility functions ready:")
print("   • retry_failed_requests() - Retry failed weather requests")
print("   • validate_weather_data() - Check data quality")
print("")
print("💡 Usage examples:")
print("   retry_failed_requests()  # Retry failed requests")
print("   validate_weather_data()  # Check data quality")

🔧 UTILITY FUNCTIONS
✅ Utility functions ready:
   • retry_failed_requests() - Retry failed weather requests
   • validate_weather_data() - Check data quality

💡 Usage examples:
   retry_failed_requests()  # Retry failed requests
   validate_weather_data()  # Check data quality


## 📋 Summary & Usage Instructions

### 🎯 **What This Notebook Does:**

1. **Analyzes Missing Data**: Compares existing weather data with required flight-based weather records
2. **Fetches Missing Records**: Uses Open-Meteo API to fetch weather data one record at a time
3. **Handles Failures Gracefully**: Implements retry logic, rate limiting, and timeout handling
4. **Saves Progress**: Automatically saves progress to avoid data loss during long runs
5. **Merges Complete Dataset**: Combines new data with existing data and updates main files

### 🚀 **How to Use:**

**Run the cells in order:**
1. **Setup & Import** - Load libraries and configure API settings
2. **Load Data** - Load existing weather, flight, and airport data
3. **Analyze Missing** - Identify what weather data is missing
4. **Setup Functions** - Prepare weather fetching functions
5. **Fetch Data** - Run the main fetching process (this takes time!)
6. **Merge & Save** - Combine and save the complete dataset
7. **Utilities** - Use helper functions as needed

### ⚡ **Key Features:**

- **Resume Capability**: Can resume from previous progress if interrupted
- **Rate Limiting**: Respects API limits with 1.1s delays between requests
- **Retry Logic**: Automatically retries failed requests with exponential backoff
- **Progress Tracking**: Shows progress and ETA during long runs
- **Error Handling**: Saves failed requests for later retry attempts
- **Data Validation**: Includes functions to validate data quality

### 📊 **Expected Results:**

- Improved weather data coverage from ~60% to near 100%
- Complete weather dataset saved as `weather_data.csv`
- Failed requests logged for potential retry
- Progress tracking and resumption capability

### ⚠️ **Important Notes:**

- **Time Required**: May take several hours depending on missing data volume
- **API Limits**: Respects Open-Meteo rate limits (10,000 requests/day)
- **Network Dependent**: Requires stable internet connection
- **Resumable**: Can be stopped and resumed using progress files